***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# # Import folder name
# df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
# df_about = df_about[df_about['Indicator'] == indicator_name]
# folder = df_about.Folder.values[0]
# print(folder)

In [ ]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = import_tab)


if estimate == 'DEC':
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = estimate)
elif sample_type == 'PUMS':
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Table Type'] == df_inputs['table'].values[0]]
    weight  = df_vars['Suggested Weight'].values[0]
else:
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
    
# Subset variables
df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set years
years_to_import = list(range(year_start, year_end+1))

def ME_split(text):
    return ",".join(text.split(',')[0:3:2])

if margin_of_error == 'Yes':
    df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)


## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type == 'ACS':
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except Exception as e: print(e)

    if margin_of_error == 'Yes':
        # list_vars = ['NAME'] + df_vars['ID'].to_list() + df_vars['Attributes'].to_list()
        list_vars = ['NAME'] + df_vars['ID_Attributes'].to_list()
    else:
        list_vars = ['NAME'] + df_vars['ID'].to_list()
    tables = df_vars['Table'].unique()

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values)) 
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = df_fips.copy()
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(tables)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if import_tab == 'PUMA':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except Exception as e: print(e)

    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS'})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)



if estimate == 'CPS':
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
    
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    
    df_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values)) 
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = df_fips.copy()
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
        
    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

***

## Importing

***

In [ ]:
# df_census = df_census_raw.copy()
# df_census = df_census[(df_census['PERRP'].isin(['40', '41'])) | (df_census['PERRP'].isin(['1', '2']))]

# df1 = df_census.copy()
# df2 = df_census.drop_duplicates(['HRHHID', 'HRHHID2', 'Year'])

# print(df1.shape)
# print(df2.shape)

In [ ]:
start_time = time.time()

## For ACS1 or ACS5 data
if sample_type == 'ACS':
  
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    
    for table in tables:
    
        print("")
        print("Table ID: " + table)
        print("")
        list_df_tables = []
    
        df_table = df_vars[df_vars['Table'] == table]
        if margin_of_error == 'Yes':
            list_table_vars = [['NAME'] + df_table['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_table['ID_Attributes'].to_list()), 20)]
        else:
            list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+45] for x in range(0, len(df_table['ID'].to_list()), 45)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        list_df_vars = []
        
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_years = []

            if import_tab == 'Counties':
                for state in list(dict_fips.keys()):
                    print('State: ' + state)
                    for year in tqdm(years_to_import):
                        try:
                            list_df_years.append(
                                query_census(api_key      = api_key
                                              , estimate  = estimate
                                              , sample    = sample_type
                                              , geography = geography
                                              , variables = variables
                                              , year      = year
                                              , state     = state
                                              , county    = dict_fips[state])
                            )
                        except Exception as e: print(e)
                df_years = pd.concat(list_df_years)
                            
            if import_tab == 'MSA':
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(api_key      = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , msa       = msa_to_import)
                        )
                    except Exception as e: print(e)
                df_years = pd.concat(list_df_years)

            list_df_vars.append(df_years)
        
        if geography == 'Tract':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'County':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'MSA':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)

        list_df_census.append(df_vars_all)
        print("All variables from table ID " + table + " have been reduced together into one table")
        print("")

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tract':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'County':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For DEC data
if estimate == 'DEC':

    print("Importing and compiling Decennial data from the Census Bureau...")
    print("")

    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties
    # reduce all tables/variables pulled into one table
    
    list_df_census = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_census.append(
                    query_census(api_key       = api_key
                                   , estimate  = estimate
                                   , sample    = sample_type
                                   , geography = geography
                                   , variables = ','.join(dict_vars[str(year)])
                                   , year      = year
                                   , state     = state
                                   , county    = dict_fips[state])
                )
            except Exception as e: print(e)
                    
    if geography == 'Tract':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'tract', 'Year']).reset_index()
    if geography == 'County':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()



## For PUMS tables
if geography == 'PUMA':
    
    print("Importing and compiling PUMS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and PUMAs
    # import all variables
    # combine all years and PUMAs
    # outer join variables onto ID fields for each geography type
    # calculate margin of error using replicate weights
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        list_df_years = []
        for year in years_to_import:
            print("")
            print('Year: ' + str(year))
            list_df_vars = []
            try:
                list_table_vars = [dict_vars[str(year)][x:x+45] for x in range(0, len(dict_vars[str(year)]), 45)]
                
                list_variables = []
                for x in list_table_vars:
                    list_variables.append(",".join(x))

                print('Querying variables...')
                for variables in tqdm(list_variables):
                    df_pums = query_census(api_key       = api_key
                                             , estimate  = estimate
                                             , sample    = sample_type
                                             , geography = geography
                                             , variables = variables + ',PUMA,SERIALNO'
                                             , year      = year
                                             , state     = state
                                             , puma      = dict_fips[state])
                    df_pums['state'] = state
                    df_pums = df_pums.drop(['public use microdata area'], axis = 1)
                    list_df_vars.append(df_pums)

                df_vars_years = ft.reduce(lambda left, right: pd.merge(left, right, on = ['state', 'SERIALNO', 'Year', 'PUMA'], how = 'outer'), list_df_vars)
                df_vars_years = df_vars_years.set_index(['state', 'SERIALNO', 'Year', 'PUMA']).reset_index()
                df_vars_years.columns = ['state', 'SERIALNO', 'Year', 'PUMA'] + dict_vars[str(np.max(years_to_import))]
                
                print('Calculating margin of error using replicate weights...')
                cols = [col for col in df_vars_years.columns if weight in col]
                df_vars_years[cols] = df_vars_years[cols].astype(int)
                cols = ['state', 'SERIALNO', 'Year', 'PUMA'] + cols
                df_me = df_vars_years[cols]
                df_me = pd.melt(df_me
                               , id_vars    = ['state', 'SERIALNO', 'Year', 'PUMA', weight]
                               , var_name   = 'replicates'
                               , value_name = 'replicate_weights')
                df_me['sq_diff'] = (df_me['replicate_weights'] - df_me[weight])**2
                df_me = df_me.groupby(['state', 'SERIALNO', 'Year', 'PUMA', weight], as_index = False)['sq_diff'].agg(sum)
                df_me['variance'] = df_me['sq_diff']*(4/80)
                df_me['SE'] = np.sqrt(df_me['variance'])
                df_me['ME'] = df_me['SE']*1.645
                df_me = df_me[['state', 'SERIALNO', 'Year', 'PUMA', weight, 'ME']]
                cols_to_drop = df_vars_years.columns[df_vars_years.columns.str.contains(weight)]
                df_vars_years = df_vars_years.drop(cols_to_drop, axis = 1)
                
                df_vars_years = df_vars_years.merge(df_me, on = ['state', 'SERIALNO', 'Year', 'PUMA'], how = 'left')
                list_df_years.append(df_vars_years)
                print('Success!')
                
            except Exception as e: print(e)
                                
    df_census_raw = pd.concat(list_df_years)


## For CPS tables
if estimate == 'CPS':

    list_df_census = []

    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                temp = query_census(api_key      = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = ','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                list_df_census.append(temp)
            except:
                pass
                
    df_census_raw = pd.concat(list_df_census)

    # merge county name onto table
    df_census_raw['state' ] = df_census_raw['state' ].astype(str).apply('{:0>2}'.format)    
    df_census_raw['county'] = df_census_raw['county'].astype(str).apply('{:0>3}'.format)    
    df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                          , left_on = ['state', 'county']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
    df_census_raw = df_census_raw.set_index(['state', 'county', 'County Name', 'Year']).reset_index()
    


print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

In [ ]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
df_census_raw.head(3)

***

## Processing

***

In [ ]:
## Make copy of data frame
df_census = df_census_raw.copy()

## For tract level data
if geography == 'Tract':

    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed+

    df_census = df_census.replace('-666666666', np.nan)
    df_census = df_census.replace('null', np.nan)
    df_census = df_census.dropna(axis = 1, how = 'all')
    
    df_census = pd.melt(df_census
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'tract', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    df_census = df_census.dropna()
    
    df_census['Total'] = df_census['Total'].apply(pd.to_numeric)
    df_census = df_census.merge(df_vars[['ID', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_census = df_census[['ID', 'County Name', 'NAME', 'state', 'Label_clean',
                     'county', 'tract', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    df_census = df_census.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
        , 'tract':'Tract ID'
    })

    if indicator_name in ['Income_1', 'Income_3']:
        df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
        df_cpi = df_cpi[['Year', 'IAF_' + str(year_end)]]

        df_census = df_census.merge(df_cpi, on = 'Year', how = 'left')
        df_census['Total'] = df_census['Total']*df_census['IAF_' + str(year_end)]          
        df_census = df_census.drop(['IAF_' + str(year_end)], axis = 1)

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if indicator_name == 'Income_3':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3 Race', 'Pop_3 Tract ACS5.xlsx'), sheet_name = 'Tracts')
        df_pop = df_pop[['NAME', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_census = df_census.merge(df_pop, on = ['NAME', 'Year', 'Race_Ethnicity'], how = 'left')
        # df_census = df_census.fillna(0)
        # df_census = df_census.dropna()
        

## For County level data
if (geography == 'County') & (estimate != 'CPS'):

    # Clean missings
    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed
    
    df_census = df_census.replace('null', np.nan)
    df_census = df_census.replace('-666666666', np.nan)
    df_census = df_census.dropna(axis = 1, how = 'all')
    
    df_census = pd.melt(df_census
                      , id_vars = ['County Name', 'NAME', 'state', 'county', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )
    
    df_census['Total'] = df_census['Total'].apply(pd.to_numeric)
    df_census = df_census.merge(df_vars[['ID', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_census = df_census[['ID', 'County Name', 'NAME', 'state', 'Label_clean',
                       'county', 'Year', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]
    df_census = df_census.rename(columns = {
        'ID':'Table ID'
        , 'state':'State FIPS'
        , 'county':'County FIPS'
    })

    if indicator_name in ['Income_1', 'Income_3']:
        df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
        df_cpi = df_cpi[['Year', 'IAF_' + str(year_end)]]

        df_census = df_census.merge(df_cpi, on = 'Year', how = 'left')
        df_census['Total'] = round(df_census['Total']*df_census['IAF_' + str(year_end)])
        df_census = df_census.drop(['IAF_' + str(year_end)], axis = 1)
        

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if indicator_name == 'Income_3':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3 Race', 'Pop_3 County ACS5.xlsx'), sheet_name = 'Counties')
        df_pop = df_pop[['County Name', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_census = df_census.merge(df_pop, on = ['County Name', 'Year', 'Race_Ethnicity'], how = 'left')
        # df_census = df_census.fillna(0)
        # df_census = df_census.dropna()

## For MSA level data
if geography == 'MSA':

    # Rename columns, make sure MSA titles are consitent with current year (sometimes the MSA name changes over time)
    # Melt data from wide to long
    # Convert imported values to numeric
    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    # Manually check column names and clean as needed
    
    df_census = df_census.replace('-666666666', np.nan)
    df_census = df_census.replace('null', np.nan)
    df_census = df_census.dropna(axis = 1, how = 'all')

    
    df_census = df_census.rename(columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})
    df_msa_map = df_census[df_census['Year'] == 2022][['NAME', 'MSA_ID']].drop_duplicates().rename(columns = {'NAME':'MSA'})
    df_census = df_census.merge(df_msa_map, on = 'MSA_ID', how = 'left')
    df_census = pd.melt(df_census
                      , id_vars = ['NAME', 'MSA', 'MSA_ID', 'Year']
                      , var_name = 'ID'
                      , value_name = 'Total'
                     )

    df_census['Total'] = df_census['Total'].apply(pd.to_numeric)
    df_census = df_census.merge(df_vars[['ID', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort']], on = 'ID', how = 'left')
    df_census = df_census[['ID', 'MSA_ID', 'MSA', 'Year', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort', 'Total']]


    if indicator_name in ['Income_1', 'Income_3']:
        df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI Inflation Adjustment Factors.xlsx'), sheet_name = 'BLS_West')
        df_cpi = df_cpi[['Year', 'IAF_' + str(year_end)]]

        df_census = df_census.merge(df_cpi, on = 'Year', how = 'left')
        df_census['Total'] = round(df_census['Total']*df_census['IAF_' + str(year_end)])
        df_census = df_census.drop(['IAF_' + str(year_end)], axis = 1)

    # Some indicators require the roll up to be weighted by population
    # The following step aligns the Race/Ethnicity mappings with the population counts workbook
    if indicator_name == 'Income_3':
        df_pop = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community'
                                          , 'Pop and Demographics', 'Pop_3 Race', 'Pop_3 MSA ACS5.xlsx'), sheet_name = 'MSA')
        df_pop = df_pop[['MSA', 'Year','Race_Ethnicity', 'Population']]
        conditions = [
                        (df_pop["Race_Ethnicity"] == 'All'                                            ),
                        (df_pop["Race_Ethnicity"] == 'American Indian or Alaska Native (NH)'          ),
                        (df_pop["Race_Ethnicity"] == 'Asian (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Black or African American (NH)'                 ),
                        (df_pop["Race_Ethnicity"] == 'Hispanic or Latino'                             ),
                        (df_pop["Race_Ethnicity"] == 'Native Hawaiian or other Pacific Islander (NH)' ),
                        (df_pop["Race_Ethnicity"] == 'White (NH)'                                     ),
                        (df_pop["Race_Ethnicity"] == 'Some other race (NH)'                           ),
                        (df_pop["Race_Ethnicity"] == 'Two or more races (NH)'                         )
                    ]
        choices = ["All", "American Indian or Alaska Native", "Asian", "Black or African American", "Hispanic or Latino",
                   "Native Hawaiian or other Pacific Islander", "White (NH)", "Some other race", "Two or more races"]
        df_pop["Race_Ethnicity"] = np.select(conditions, choices)
        df_census = df_census.merge(df_pop, on = ['MSA', 'Year', 'Race_Ethnicity'], how = 'left')
        df_census = df_census.dropna()



if sample_type in ['PUMS', 'FOODSEC']:

    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data
    groups = list(df_inputs['groups'].dropna().values)
    weight = df_vars['Suggested Weight'].values[0]

    if sample_type == 'PUMS':
        df_census['PUMA'] = df_census['PUMA'].astype(str).apply('{:0>5}'.format)

    if sample_type == 'FOODSEC':
        cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
        df_census = df_census[cols]

    ## For non-integers only
    for group in groups:
        df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
        
    df_vars   ['Value1'] = df_vars   ['Value1'].astype(str).apply('{:0>2}'.format)
    df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)

    df_census.loc[df_census[weight] == '', weight] = 0
    if sample_type == 'PUMS':
        df_census[weight] = df_census[weight].astype(int)
    df_census[groups] = df_census[groups].astype("string")

    df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
                                           , columns = 'ID2'
                                           , values = 'Description2'
                                           , aggfunc = lambda x: x).reset_index()

    cols = ['Year', 'Value1'] + groups
    df_vars2 = df_vars2[cols]

    list_values = []
    for group in groups:
        list_values = list_values + list(df_census[group].values)
    set_values = set(list_values)
    
    df_vars2 = df_vars2[df_vars2['Value1'].isin(set_values)]
    df_vars2 = df_vars2.add_suffix('_desc').rename(columns = {'Value1_desc':'Value1', 'Year_desc':'Year'})

    for col in cols[2:]:
        # df_census = df_census.merge(df_vars2[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'inner')
        df_census = df_census.merge(df_vars2[['Value1', col+'_desc', 'Year']], left_on = [col, 'Year'], right_on = ['Value1', 'Year'], how = 'left')
        df_census = df_census.drop(['Value1'], axis = 1)

    # if sample_type == 'FOODSEC':
    #     df_census.loc[df_census['PRDTHSP_desc'].str.contains(','), 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
        
    # For non-integers only


# view
df_census.head(3)

In [ ]:
# unique(list(df_census.PRDTHSP_desc.values))
# df_census['PTDTRACE'].unique()

# df_census.loc[df_census['PRDTHSP'] == '-1', 'PRDTHSP_desc'] = 'Not Hispanic or Latino'
# df_census['PRDTHSP_desc'].unique()

# df_census[df_census['PRDTHSP'] == '01']

In [ ]:
# df_census = df_census_raw.copy()
# cols = ['state', 'county', 'County Name', 'Year'] + dict_vars[str(year)]
# df_census = df_census[cols]

# ## For non-integers only
# for group in groups:
#     df_census[group] = df_census[group].astype(str).apply('{:0>2}'.format)
    
# df_vars['Value1'] = df_vars['Value1'].astype(str).apply('{:0>2}'.format)
# df_census ['state' ] = df_census ['state' ].astype(str).apply('{:0>2}'.format)
# df_census.loc[df_census[weight] == '', weight] = 0
# df_census[weight] = df_census[weight].astype(int)
# df_census[groups] = df_census[groups].astype("string")
# df_vars2 = df_vars.pivot_table(index = ['Year', 'Value1']
#                                        , columns = 'ID2'
#                                        , values = 'Description2'
#                                        , aggfunc = lambda x: x).reset_index()
# cols = ['Year', 'Value1'] + groups
# df_vars2 = df_vars2[cols]

In [ ]:
if margin_of_error == 'Yes':
    path_out_me = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
    df_census_me = df_census.copy()
    
    if geography == 'Tract':
        df_me = df_census_me[df_census_me['Variable'].isna()]
        df_census_me = df_census_me.dropna()
        df_me = df_me[['Table ID', 'Tract ID', 'County Name', 'Year', 'Total']].rename(columns = {'Total':'ME'})
        df_me['Table ID'] = df_me['Table ID'].apply(lambda s : re.sub("M", "E", s))
        df_census_me = df_census_me.merge(df_me, on = ['Table ID', 'Tract ID', 'County Name', 'Year'], how = 'left')
        # df_census_me = df_census_me[['Table ID', 'County Name', 'Year', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort', 'Total', 'ME']]
        df_census_me['Year'] = df_census_me['Year'].astype(str)
        df_census = df_census_me.copy()
        # df_census_me.to_excel(os.path.join(path_out_me, "".join([indicator_name, '_', geography, '_ME_', estimate, '.xlsx'])), index = False)
    
    if geography == 'County':
        df_me = df_census_me[df_census_me['Variable'].isna()]
        df_census_me = df_census_me.dropna()
        df_me = df_me[['Table ID', 'County Name', 'Year', 'Total']].rename(columns = {'Total':'ME'})
        df_me['Table ID'] = df_me['Table ID'].apply(lambda s : re.sub("M", "E", s))
        df_census_me = df_census_me.merge(df_me, on = ['Table ID', 'County Name', 'Year'], how = 'left')
        # df_census_me = df_census_me[['Table ID', 'County Name', 'Year', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort', 'Total', 'ME']]
        df_census_me['Year'] = df_census_me['Year'].astype(str)
        df_census = df_census_me.copy()
        # df_census_me.to_excel(os.path.join(path_out_me, "".join([indicator_name, '_', geography, '_ME_', estimate, '.xlsx'])), index = False)

    if geography == 'MSA':
        df_me = df_census_me[df_census_me['Variable'].isna()]
        df_census_me = df_census_me.dropna()
        df_me = df_me[['ID', 'MSA', 'Year', 'Total']].rename(columns = {'Total':'ME'})
        df_me['ID'] = df_me['ID'].apply(lambda s : re.sub("M", "E", s))
        df_census_me = df_census_me.merge(df_me, on = ['ID', 'MSA', 'Year'], how = 'left')
        df_census_me = df_census_me[['ID', 'MSA', 'Year', 'Label_clean', 'Variable', 'Race_Ethnicity', 'Sort', 'Total', 'ME']]
        df_census = df_census_me.copy()
        # df_census_me.to_excel(os.path.join(path_out_me, "".join([indicator_name, '_', geography, '_ME_', estimate, '.xlsx'])), index = False)


df_census


In [ ]:
## For ACS tables
if sample_type == 'ACS':
    
    # Create "Categorical" race/ethnicity field for sorting
    # Sort by geography, variable mapping, and race/ethnicity
    # sort and then remove categorical field

    df_census['Race_Ethnicity_sort'] = pd.Categorical(df_census['Race_Ethnicity'], ['All'
                                                                     , 'American Indian or Alaska Native'
                                                                     , 'American Indian or Alaska Native (NH)'
                                                                     , 'Asian'
                                                                     , 'Asian (NH)'
                                                                     , 'Black or African American'
                                                                     , 'Black or African American (NH)'
                                                                     , 'Hispanic or Latino'
                                                                     , 'Native Hawaiian or other Pacific Islander'
                                                                     , 'Native Hawaiian or other Pacific Islander (NH)'
                                                                     , 'White'
                                                                     , 'White (NH)'
                                                                     , 'Some other race'
                                                                     , 'Some other race (NH)'
                                                                     , 'Two or more races'
                                                                     , 'Two or more races (NH)'
                                                                             ])
    if geography == 'Tract':
        df_census = df_census.sort_values(by = ['NAME'       , 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = [True, False, True, True])
    if geography == 'County':
        df_census = df_census.sort_values(by = ['County Name', 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = [True, False, True, True])
    if geography == 'MSA':
        df_census = df_census.sort_values(by = ['MSA'        , 'Year', 'Race_Ethnicity_sort', 'Sort'], ascending = [True, False, True, True])
    df_census = df_census.drop(['Race_Ethnicity_sort', 'Sort'], axis = 1)


## For PUMS tables
if geography == 'PUMA':

    # Remove rows with missing values
    # Sort by PUMA, Year, then by each group
    # Only keep description mappings, remove the original PUMS values
    # Rollup using suggested weight field
    # TODO: merge on PUMA name field

    df_census = df_census.dropna()
    
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums[df_fips_pums['STATEFP'].isin(list(dict_fips.keys()))]
    df_fips_pums = df_fips_pums[['STATEFP', 'PUMA5CE', 'PUMA NAME', 'COUNTYFP', 'Years']].rename(columns = {'PUMA5CE':'PUMA', 'STATEFP':'state'}).drop_duplicates()

    df_census1 = df_census[df_census['Year'].isin(sequence(2012, 2021, 1))]
    df_census2 = df_census[df_census['Year'].isin(sequence(2022, 2031, 1))]
    
    df_census1 = df_census1.merge(df_fips_pums[df_fips_pums['Years'] == '2012-2021'], on = ['state', 'PUMA'], how = 'left')
    df_census2 = df_census2.merge(df_fips_pums[df_fips_pums['Years'] == '2022-2031'], on = ['state', 'PUMA'], how = 'left')
    df_census = pd.concat([df_census1, df_census2])
    df_census = df_census.drop('Years', axis = 1)
    
    df_census = df_census.sort_values(
        ['PUMA', 'Year'] + groups
        , ascending = [True, False] + [item in groups for item in groups]
    )

    ## For non-integers only
    df_census = df_census.drop(groups, axis = 1)

    if 'HISP' in groups:
        df_census.loc[df_census['HISP_desc'] == 'Hispanic or Latino', 'RAC1P_desc'] = 'Hispanic or Latino'
        df_census = df_census.drop('HISP_desc', axis = 1)

    if 'HHLDRHISP' in groups:
        df_census.loc[df_census['HHLDRHISP_desc'] == 'Hispanic or Latino', 'HHLDRRAC1P_desc'] = 'Hispanic or Latino'
        df_census = df_census.drop('HHLDRHISP_desc', axis = 1)
    ## For non-integers only
    df_census = df_census.drop(['PUMA', 'PUMA NAME', 'SERIALNO'], axis = 1)
    df_census = df_census.groupby(list(df_census.drop([weight, 'ME'], axis = 1).columns), as_index = False, sort = False).agg(Total = ('PWGTP', 'sum'), ME = ('ME', sqrtsumsq))

    # col = df_census.pop('PUMA NAME')
    # df_census.insert(1, 'PUMA NAME', col)

    col = df_census.pop('state')
    df_census.insert(0, 'state', col)

# View
df_census.head(3)

In [ ]:
if geography == 'Tract':
    
    # Create MPO and MSA groupings
    # Merge groupings
    # reorder columns
    # fill missing values (represent a population of 0)
    df_mpo = df_fips[['County Name', 'MPO']]
    df_census = df_census.merge(df_mpo, on = ['County Name'], how = 'left')
    if indicator_name == 'Income_3':
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name', 'Tract ID',
                'County FIPS', 'NAME', 'Year', 'Race_Ethnicity', 'Variable', 'Total', 'Population']
    else:
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name', 'Tract ID',
            'County FIPS', 'NAME', 'Year', 'Race_Ethnicity', 'Variable', 'Total', 'ME']
    df_census = df_census[cols]
    df_census['Total'] = df_census['Total'].fillna(0)
    

    # Income_3 needs to be weighted by population
    if indicator_name == 'Income_3':
        # Fill missings with 0, then 1 to make sure nothing gets removed if population is 0
        # create weighted average lambda function
        # roll up to different geographies using population weighted average
        df_census['Population'] = df_census['Population'].fillna(0)
        df_census['Population'] = df_census['Population'].replace(0, 1)
        df_census1 = df_census.groupby(['State FIPS', 'MPO', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Population = ('Population', 'sum'), Total = ('Total', wm))
        df_census1 = df_census1.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])

    # For all other indicators
    else:
        df_census1 = df_census.groupby(['State FIPS', 'MPO', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
        df_census1['ME_ratio'] = df_census1['ME']/df_census1['Total']*100
        df_census1 = df_census1.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])

    
    # Reshape data to wide format
    df_census2 = df_census1.pivot_table(index = ['State FIPS', 'MPO', 'County FIPS', 'Tract ID', 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_census2 = df_census2.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])

    # Replace infinite values with NaN
    # missing values represent a population of 0
    df_census1 = df_census1.replace([np.inf, -np.inf, 0], np.nan)
    df_census2 = df_census2.fillna(0)

    ## Check if we want to calculate proportions
    if percentages == 'Yes':
        if num_vars == 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1[df_census1['Race_Ethnicity'] != 'All'].groupby(['NAME', 'Year'])['Total'].transform('sum')
        if num_vars > 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1.groupby(['NAME', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')

        # Reshape data to wide format
        df_census2_pct = df_census1.pivot_table(index = ['State FIPS', 'MPO', 'County FIPS', 'Tract ID', 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_census2_pct = df_census2_pct.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])
        
        # missing values represent a population of 0
        df_census2_pct = df_census2_pct.fillna(0)


    # # Calculate Five-Number Summaries
    # df_census1_summaries = df_census1[df_census1['Race_Ethnicity'] == 'All'].groupby(['County Name', 'Variable', 'Year'], as_index = False)['Total']\
    #                                                                 .describe()\
    #                                                                 .rename(columns = {'25%':'Q1', '50%':'median', '75%':'Q3'})
    # # Set workbook name
    # name_output_summaries_xlsx = [indicator_name, ' MPO ', estimate, ' Summaries.xlsx']
    # name_output_summaries_xlsx = "".join(name_output_summaries_xlsx)
    
    # # Set file path for exporting
    # path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
    # # df_census1_summaries.to_excel(os.path.join(path_out_xlsx, name_output_summaries_xlsx), index = False)


if geography == 'County':
    # Create MPO and MSA groupings
    # Merge groupings
    # reorder columns
    # fill missing values (represent a population of 0)
    
    df_mpo = df_fips[['County Name', 'MPO']]
    df_census = df_census.merge(df_mpo, on = ['County Name'], how = 'left')
    if indicator_name == 'Income_3':
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name',
                'County FIPS', 'NAME', 'Year', 'Race_Ethnicity', 'Variable', 'Population', 'Total', 'ME']
    else:
        cols = ['Table ID', 'State FIPS', 'MPO', 'County Name', 
            'County FIPS', 'NAME', 'Year', 'Race_Ethnicity', 'Variable', 'Total', 'ME']
    df_census = df_census[cols]
    df_census['Total'] = df_census['Total'].fillna(0)

    
    # Income_3 needs to be weighted by population
    if indicator_name == 'Income_3':
        # Fill missings with 0, then 1 to make sure nothing gets removed if population is 0
        # create weighted average lambda function
        # roll up to different geographies using population weighted average
        df_census['Population'] = df_census['Population'].fillna(0)
        df_census['Population'] = df_census['Population'].replace(0, 1)
        df_census.loc[df_census['ME'] < 0, 'ME'] = np.nan
        df_census1 = df_census.groupby(['State FIPS', 'MPO', 'County FIPS', 'County Name', 'NAME', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Population = ('Population', 'sum'), Total = ('Total', wm), ME = ('ME', sqrtsumsq))  
        df_census1['ME_ratio'] = df_census1['ME']/df_census1['Total']*100
        df_census1 = df_census1.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])
        df_mpo1 = df_census.groupby(['State FIPS', 'MPO', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Population = ('Population', 'sum'), Total = ('Total', wm), ME = ('ME', sqrtsumsq))  
        df_mpo1['ME_ratio'] = df_mpo1['ME']/df_mpo1['Total']*100
        df_mpo1 = df_mpo1.sort_values(['MPO', 'Year'], ascending = [True, False])

    # For all other indicators
    else:
        df_census.loc[df_census['ME'] < 0, 'ME'] = np.nan
        df_census1 = df_census.groupby(['State FIPS', 'MPO', 'County FIPS', 'County Name', 'NAME', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
        df_census1['ME_ratio'] = df_census1['ME']/df_census1['Total']*100
        df_census1 = df_census1.sort_values(['MPO', 'NAME', 'Year'], ascending = [True, True, False])
        df_mpo1 = df_census.groupby(['State FIPS', 'MPO', 'Year', 'Race_Ethnicity', 'Variable']
                                 , as_index = False, sort = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
        df_mpo1['ME_ratio'] = df_mpo1['ME']/df_mpo1['Total']*100
        df_mpo1 = df_mpo1.sort_values(['MPO', 'Year'], ascending = [True, False])


    # Reshape data to wide format
    df_census2 = df_census1.pivot_table(index = ['State FIPS', 'MPO', 'County FIPS', 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_census2 = df_census2.sort_values(['MPO', 'County FIPS', 'Year'], ascending = [True, True, False])
    df_mpo2 = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_mpo2 = df_mpo2.sort_values(['MPO', 'Year'], ascending = [True, False])

    # Replace infinite values with NaN
    df_census1 = df_census1.replace([np.inf, -np.inf, 0], np.nan)
    df_mpo1    = df_mpo1   .replace([np.inf, -np.inf, 0], np.nan)
    
    # missing values represent a population of 0
    df_census2 = df_census2.fillna(0)
    df_mpo2    = df_mpo2   .fillna(0)

    
    ## Check if we want to calculate proportions
    if percentages == 'Yes':

        if num_vars == 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1[df_census1['Race_Ethnicity'] != 'All'].groupby(['NAME', 'Year'])['Total'].transform('sum')
            df_mpo1   ['Percentage'] = 100*df_mpo1   ['Total'] / df_mpo1   [df_mpo1   ['Race_Ethnicity'] != 'All'].groupby(['MPO' , 'Year'])['Total'].transform('sum')
        if num_vars > 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1.groupby(['NAME', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')
            df_mpo1   ['Percentage'] = 100*df_mpo1   ['Total'] / df_mpo1   .groupby(['MPO' , 'Year', 'Race_Ethnicity'])['Total'].transform('sum')

        # Reshape data to wide format
        df_census2_pct = df_census1.pivot_table(index = ['State FIPS', 'MPO', 'County FIPS',  'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_census2_pct = df_census2_pct.sort_values(['MPO', 'County FIPS', 'Year'], ascending = [True, True, False])
        df_mpo2_pct = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_mpo2_pct = df_mpo2_pct.sort_values(['MPO', 'Year'], ascending = [True, False])
        
        # missing values represent a population of 0
        df_census2_pct = df_census2_pct.fillna(0)
        df_mpo2_pct = df_mpo2_pct.fillna(0)


if geography == 'MSA':
    
    # Groupings roll up
    df_census.loc[df_census['ME'] < 0, 'ME'] = np.nan
    df_census1 = df_census.groupby(['MSA', 'Year', 'Race_Ethnicity', 'Variable']
                             , as_index = False, sort = False).agg(Total = ('Total', 'sum'), ME = ('ME', sqrtsumsq))
    df_census1['ME_ratio'] = df_census1['ME']/df_census1['Total']*100
    df_census1 = df_census1.sort_values(['MSA', 'Year'], ascending = [True, False])

    # Reshape data to wide format
    df_census2 = df_census1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                       , columns = 'Variable'
                                       , values = 'Total').reset_index()
    df_census2 = df_census2.sort_values(['MSA', 'Year'], ascending = [True, False])

    # Replace infinite values with NaN
    df_census1 = df_census1.replace([np.inf, -np.inf, 0], np.nan)
    
    # missing values represent a population of 0
    df_census2 = df_census2.fillna(0)

    ## Check if we want to calculate proportions
    if percentages == 'Yes':

        # Estimate proportions by groupings
        if num_vars == 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1[df_census1['Race_Ethnicity'] != 'All'].groupby(['MSA', 'Year'])['Total'].transform('sum')
        if num_vars > 1:
            df_census1['Percentage'] = 100*df_census1['Total'] / df_census1.groupby(['MSA', 'Year', 'Race_Ethnicity'])['Total'].transform('sum')
            
        # Reshape data to wide format
        df_census2_pct = df_census1.pivot_table(index = ['MSA', 'Year', 'Race_Ethnicity']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_census2_pct = df_census2_pct.sort_values(['MSA', 'Year'], ascending = [True, False])
        
print("Done!  Probably good")

In [ ]:
if indicator_name == 'Income_3':
    df_mpo_wm = df_mpo1[df_mpo1['Race_Ethnicity'] == 'All'].reset_index(drop = True)

    df_census1 = df_census1.merge(df_mpo_wm[['Year', 'Total']].rename(columns = {'Total':'Regional Median Household Income'}), on = ['Year'], how = 'left')
    df_mpo1 = df_mpo1.merge(df_mpo_wm[['Year', 'Total']].rename(columns = {'Total':'Regional Median Household Income'}), on = ['Year'], how = 'left')

    df_census1['Percent of Regional Median Household Income'] = 100*(df_census1['Total']/df_census1['Regional Median Household Income'])
    df_mpo1['Percent of Regional Median Household Income'] = 100*(df_mpo1['Total']/df_mpo1['Regional Median Household Income'])

    df_census1 = df_census1.rename(columns = {'Total':'Median Household Income', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
    df_mpo1 = df_mpo1.rename(columns = {'Total':'Median Household Income', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})


if indicator_name == 'Income_1':
    df_census1 = df_census1.rename(columns = {'Total':'Regional Median Income', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
    if geography == 'County':
        df_mpo1 = df_mpo1.rename(columns = {'Total':'Regional Median Income', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})


if indicator_name in ['Cost_5', 'Income_2', 'Broadband_2']:
    df_census1 = df_census1.rename(columns = {'Total':'Households', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
    if geography == 'County':
        df_mpo1 = df_mpo1.rename(columns = {'Total':'Households', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
if indicator_name in ['Cost_3']:
    df_census1 = df_census1.rename(columns = {'Total':'Housing Units', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
    if geography == 'County':
        df_mpo1 = df_mpo1.rename(columns = {'Total':'Housing Units', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})


if indicator_name in ['Pop_3', 'Pop_4', 'Edu_1', 'Labor_1', 'Health_2', 'Income_4', 'Commute_1']:
    df_census1 = df_census1.rename(columns = {'Total':'Population', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})
    if geography == 'County':
        df_mpo1 = df_mpo1.rename(columns = {'Total':'Population', 'ME':'Margin of Error', 'ME_ratio':'Margin of Error Ratio'})


if geography == 'PUMA':
    if 'HISP_desc' in df_census.columns:
        df_census = df_census.drop(['HISP_desc'], axis = 1)



***

## Exporting

***

In [ ]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tract':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_census1.to_excel(writer, index = False, sheet_name = 'Tracts'     )
        df_census2.to_excel(writer, index = False, sheet_name = 'Tracts wide')
        if percentages == 'Yes':
            df_census2_pct.to_excel(writer, index = False, sheet_name = 'Tracts wide pct')

if geography == 'County':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_census1.to_excel(writer, index = False, sheet_name = 'Counties'     )
        df_census2.to_excel(writer, index = False, sheet_name = 'Counties wide')
        if percentages == 'Yes':
            df_census2_pct.to_excel(writer, index = False, sheet_name = 'Counties wide pct')
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_MPO_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_MPO_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO'     )
        df_mpo2.to_excel(writer, index = False, sheet_name = 'MPO wide')
        if percentages == 'Yes':
            df_mpo2_pct.to_excel(writer, index = False, sheet_name = 'MPO wide pct')
             

if geography == 'MSA':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_census1.to_excel(writer, index = False, sheet_name = 'MSA'     )
        df_census2.to_excel(writer, index = False, sheet_name = 'MSA wide')
        if percentages == 'Yes':
            df_census2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')


if geography == 'PUMA':

    # df_census.columns = [re.sub('_desc', '', col) for col in df_census.columns]
    # df_census_csv = df_census.rename(columns = {'Year':'year', 'state': 'State FIPS'})

    # df_census_csv.to_csv(os.path.join(path_out_csv, name_output_PUMA_csv), index = False)

    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        df_census.to_excel(writer, index = False, sheet_name = 'PUMS')

print('')
print("Successfully exported")